# MaskGen Token Regret Critic (Single-Notebook DDP Training)


## Setup: Import and Load

In [ ]:
import os
import sys
import importlib
import torch
import open_clip
import matplotlib.pyplot as plt
from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ

# Resolve the repo root robustly so notebook imports and local checkpoints stay aligned.
PROJECT_ROOT_CANDIDATES = [
    os.path.abspath('.'),
    os.path.dirname(os.path.abspath('.')),
    os.path.dirname(os.path.dirname(os.path.abspath('.'))),
]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(path, 'token_regrate'))),
    os.path.abspath('.'),
)
HF_CACHE_DIR = os.path.abspath(os.environ.get('HF_CACHE_DIR', os.path.join(PROJECT_ROOT, 'hf_cache')))
LOCAL_FILES_ONLY = bool(HF_CACHE_DIR)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
os.environ['OPENCLIP_CACHE_DIR'] = HF_CACHE_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Project root:', PROJECT_ROOT)
print('HF cache:', HF_CACHE_DIR)
print('LOCAL_FILES_ONLY:', LOCAL_FILES_ONLY)

# Add project root to path for imports and make relative checkpoints/data paths stable.
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


import token_regrate.config as token_regret_config
import token_regrate.train_token_regret_ddp as token_regret_train
token_regret_config = importlib.reload(token_regret_config)
token_regret_train = importlib.reload(token_regret_train)
get_config = token_regret_config.get_config

# Import from the reloaded training script - shared training and inference utilities.
from token_regrate.train_token_regret_ddp import *

def _resolve_project_path(path_value):
    path_value = os.path.expanduser(str(path_value))
    return path_value if os.path.isabs(path_value) else os.path.abspath(os.path.join(PROJECT_ROOT, path_value))

def _resolve_pretrained_path(path_value, cache_root=HF_CACHE_DIR):
    path_value = os.path.expanduser(str(path_value))
    if os.path.isabs(path_value):
        return path_value

    normalized_path = os.path.normpath(path_value)
    cache_relative_path = normalized_path
    cache_prefix = f'hf_cache{os.sep}'
    if normalized_path.startswith(cache_prefix):
        cache_relative_path = normalized_path.split(os.sep, 1)[1]

    candidates = [
        os.path.abspath(normalized_path),
        os.path.abspath(os.path.join(PROJECT_ROOT, normalized_path)),
        os.path.abspath(os.path.join(cache_root, cache_relative_path)),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]

def load_pretrained_stack(model_cfg=None):
    model_cfg = get_config().model if model_cfg is None else model_cfg
    tok_dir = _resolve_pretrained_path(model_cfg.tokenizer_path)
    gen_dir = _resolve_pretrained_path(model_cfg.generator_path)
    clip_name = str(model_cfg.clip_name)
    clip_pretrained = str(getattr(model_cfg, 'clip_pretrained', 'openai'))
    clip_force_quick_gelu = bool(getattr(model_cfg, 'clip_force_quick_gelu', True))

    print('Loading TA-TiTok tokenizer...')
    tatitok_vq_tokenizer = TATiTok.from_pretrained(pretrained_model_name_or_path=tok_dir, cache_dir=tok_dir)
    tatitok_vq_tokenizer.eval()
    tatitok_vq_tokenizer.requires_grad_(False)

    print('Loading MaskGen-VQ generator...')
    maskgen_vq_generator = MaskGen_VQ.from_pretrained(pretrained_model_name_or_path=gen_dir, cache_dir=gen_dir)
    maskgen_vq_generator.eval()
    maskgen_vq_generator.requires_grad_(False)

    print('Loading CLIP text encoder...')
    clip_encoder, _, _ = open_clip.create_model_and_transforms(
        clip_name, pretrained=clip_pretrained, force_quick_gelu=clip_force_quick_gelu
    )
    del clip_encoder.visual
    clip_tokenizer = open_clip.get_tokenizer(clip_name)
    clip_encoder.transformer.batch_first = False
    clip_encoder.eval()
    clip_encoder.requires_grad_(False)

    print('Pretrained stack ready.')
    return tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder


tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder = load_pretrained_stack()
maskgen_vq_generator = maskgen_vq_generator.to(device)
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(device)
clip_encoder = clip_encoder.to(device)
print('Generator device:', next(maskgen_vq_generator.parameters()).device)
print('Tokenizer device:', next(tatitok_vq_tokenizer.parameters()).device)
print('CLIP text encoder device:', next(clip_encoder.parameters()).device)


## Init all parameters for training and inference time

In [ ]:
# Load notebook defaults directly from token_regrate/config.py.
CONFIG = get_config()
INF_CFG = CONFIG.inference
TRAIN_CFG = CONFIG.training
DATA_CFG = CONFIG.dataset
EXPERIMENT_CFG = CONFIG.experiment
RUNTIME_CFG = CONFIG.runtime
MODEL = CONFIG.model
NOTEBOOK_GENERATE_KWARGS = {
    'guidance_scale': float(TRAIN_CFG.train_guidance_scale),
    'randomize_temperature': float(TRAIN_CFG.train_randomize_temperature),
    'aesthetic_score': float(TRAIN_CFG.train_aesthetic_score),
    'num_sample_steps': int(MODEL.sample_steps),
    'remask_ratio': float(INF_CFG.remask_ratio),
    'refine_loops': int(INF_CFG.refine_loops),
    'refine_start_step': int(INF_CFG.refine_start_step),
    'repair_greedy': bool(INF_CFG.repair_greedy),
}

print('Notebook config loaded from token_regrate/config.py')
print('Inference steps:', MODEL.sample_steps)
print('Training epochs:', TRAIN_CFG.num_epochs)
print('Training seed:', TRAIN_CFG.seed)
print('Training batch size:', TRAIN_CFG.per_gpu_batch_size)
print('Max training images:', getattr(TRAIN_CFG, 'max_train_images', 0))
print('Training output dir:', EXPERIMENT_CFG.output_dir)
print('Learning rate:', TRAIN_CFG.learning_rate)
print('Counterfactual rollout steps:', TRAIN_CFG.counterfactual_rollout_steps)
print('Counterfactual utility:', getattr(TRAIN_CFG, 'counterfactual_utility', 'token_ce'))
print('Counterfactual window radius:', getattr(TRAIN_CFG, 'counterfactual_window_radius', 0))
print('Counterfactual contrast negatives:', getattr(TRAIN_CFG, 'counterfactual_contrast_negatives', 2))
print('Counterfactual contrast temperature:', getattr(TRAIN_CFG, 'counterfactual_contrast_temperature', 1.0))
print('Counterfactual contrast mode:', getattr(TRAIN_CFG, 'counterfactual_contrast_mode', 'nce'))
print('Counterfactual repair greedy:', getattr(TRAIN_CFG, 'counterfactual_repair_greedy', None))
print('Critic prompt gap top-k:', getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 0))
print('Regret target transform:', getattr(TRAIN_CFG, 'regret_target_transform', None))
print('Inference selection: pure top-k (no score threshold)')
print('Train repair greedy:', getattr(TRAIN_CFG, 'train_repair_greedy', None))
print('Fixed rollout step:', getattr(TRAIN_CFG, 'fixed_rollout_step', None))
print('Use target replay:', getattr(TRAIN_CFG, 'use_target_critic_replay', None))
print('Lambda rank:', TRAIN_CFG.lambda_rank)
print('Resume checkpoint:', repr(str(RUNTIME_CFG.resume_checkpoint)))
print('DDP only:', bool(TRAIN_CFG.ddp_only))


## Critic Training Entry Cell

Official mode: train only the critic head from the provided dataset source.


In [ ]:
import os, torch
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
print("TOKEN_REGRET_NPROC:", os.environ.get("TOKEN_REGRET_NPROC"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
import subprocess
trained_critic = None

token_regret_nproc = os.environ.get('TOKEN_REGRET_NPROC')
if not torch.cuda.is_available():
    nproc_per_node = 1
else:
    visible_gpu_count = torch.cuda.device_count()
    # NCCL fails during communicator allocation on this host; use two ranks with Gloo by default.
    requested_nproc = int(token_regret_nproc) if token_regret_nproc else min(2, visible_gpu_count)
    if requested_nproc > visible_gpu_count:
        print(f'Reducing TOKEN_REGRET_NPROC={requested_nproc} to visible_gpu_count={visible_gpu_count}.')
    if token_regret_nproc is None and visible_gpu_count > 1:
        print(f'Defaulting TOKEN_REGRET_NPROC to {requested_nproc} with Gloo backend because multi-GPU NCCL init fails on this host.')
    nproc_per_node = max(1, min(requested_nproc, visible_gpu_count))
# if str(getattr(TRAIN_CFG, 'counterfactual_utility', '')).endswith('_contrast'):
#     max_train_images_for_contrast = int(getattr(TRAIN_CFG, 'max_train_images', 0))
#     if max_train_images_for_contrast > 0:
#         max_safe_nproc = max(1, max_train_images_for_contrast // 2)
#         if nproc_per_node > max_safe_nproc:
#             print(f'Contrastive target needs at least two prompts per rank; reducing nproc_per_node from {nproc_per_node} to {max_safe_nproc}.')
#             nproc_per_node = max_safe_nproc
#         if (max_train_images_for_contrast + nproc_per_node - 1) // nproc_per_node < 2:
#             raise RuntimeError('Contrastive counterfactual training needs at least two prompts per local rank.')
ddp_cmd = [
    sys.executable,
    '-m', 'torch.distributed.run',
    f'--nproc_per_node={nproc_per_node}',
    'token_regrate/train_token_regret_ddp.py',
    '--num-epochs', str(TRAIN_CFG.num_epochs),
    '--max-train-images', str(getattr(TRAIN_CFG, 'max_train_images', 0)),
    '--per-gpu-batch-size', str(TRAIN_CFG.per_gpu_batch_size),
    '--learning-rate', str(TRAIN_CFG.learning_rate),
    '--counterfactual-rollout-steps', str(TRAIN_CFG.counterfactual_rollout_steps),
    '--counterfactual-utility', str(getattr(TRAIN_CFG, 'counterfactual_utility', 'token_ce')),
    '--counterfactual-window-radius', str(getattr(TRAIN_CFG, 'counterfactual_window_radius', 0)),
    '--counterfactual-contrast-negatives', str(getattr(TRAIN_CFG, 'counterfactual_contrast_negatives', 2)),
    '--counterfactual-contrast-temperature', str(getattr(TRAIN_CFG, 'counterfactual_contrast_temperature', 1.0)),
    '--counterfactual-contrast-mode', str(getattr(TRAIN_CFG, 'counterfactual_contrast_mode', 'nce')),
    '--critic-prompt-gap-topk', str(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8)),
    '--regret-target-transform', str(getattr(TRAIN_CFG, 'regret_target_transform', 'tanh')),
    '--target-critic-ema-decay', str(TRAIN_CFG.target_critic_ema_decay),
    '--fixed-rollout-step', str(getattr(TRAIN_CFG, 'fixed_rollout_step', -1)),
    '--lambda-rank', str(TRAIN_CFG.lambda_rank),
    '--rank-margin', str(TRAIN_CFG.rank_margin),
    '--rank-gap-threshold', str(TRAIN_CFG.rank_gap_threshold),
    '--train-remask-ratio', str(TRAIN_CFG.train_remask_ratio),
    '--train-refine-start-step', str(TRAIN_CFG.train_refine_start_step),
    '--counterfactual-chunk-size', str(TRAIN_CFG.counterfactual_chunk_size),
    '--refine-loops', str(TRAIN_CFG.refine_loops),
    '--save-every', str(TRAIN_CFG.save_every),
    '--log-every', str(TRAIN_CFG.log_every),
    '--seed', str(TRAIN_CFG.seed),
    '--output-dir', str(EXPERIMENT_CFG.output_dir),
    '--train-data-source', str(DATA_CFG.source),
    '--train-dataset-mode', str(DATA_CFG.mode),
    '--cc12m-cache-dir', str(DATA_CFG.cc12m_cache_dir),
    '--stream-prefetch-batches', str(DATA_CFG.stream_prefetch_batches),
    '--cc12m-loader-workers', str(DATA_CFG.cc12m_loader_workers),
    '--cc12m-loader-max-pending', str(DATA_CFG.cc12m_loader_max_pending),
]
if int(getattr(TRAIN_CFG, 'max_steps', 0)) > 0:
    ddp_cmd += ['--max-steps', str(TRAIN_CFG.max_steps)]
counterfactual_repair_greedy = bool(getattr(TRAIN_CFG, 'counterfactual_repair_greedy', getattr(INF_CFG, 'repair_greedy', False)))
train_repair_greedy = bool(getattr(TRAIN_CFG, 'train_repair_greedy', getattr(TRAIN_CFG, 'counterfactual_repair_greedy', getattr(INF_CFG, 'repair_greedy', False))))
if counterfactual_repair_greedy:
    ddp_cmd += ['--counterfactual-repair-greedy']
else:
    ddp_cmd += ['--no-counterfactual-repair-greedy']
if train_repair_greedy:
    ddp_cmd += ['--train-repair-greedy']
else:
    ddp_cmd += ['--no-train-repair-greedy']
if bool(DATA_CFG.disable_cc12m_cache):
    ddp_cmd += ['--disable-cc12m-cache']
resume_checkpoint = str(getattr(RUNTIME_CFG, 'resume_checkpoint', '')).strip()
if resume_checkpoint:
    ddp_cmd += ['--resume-checkpoint', resume_checkpoint]

ddp_env = os.environ.copy()
dist_backend = os.environ.get('TOKEN_REGRET_DIST_BACKEND') or ('gloo' if nproc_per_node > 1 else str(TRAIN_CFG.dist_backend))
ddp_env['DIST_BACKEND'] = dist_backend
# Keep NCCL workaround flags available when TOKEN_REGRET_DIST_BACKEND=nccl is explicitly requested.
ddp_env['NCCL_DEBUG'] = 'INFO'
ddp_env['NCCL_P2P_DISABLE'] = '1'
ddp_env['NCCL_IB_DISABLE'] = '1'
ddp_env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
ddp_env['NCCL_CUMEM_ENABLE'] = '0'
ddp_env['NCCL_CUMEM_HOST_ENABLE'] = '0'
print('Launching DDP training:', ' '.join(ddp_cmd))
print('DDP environment:', {k: ddp_env.get(k) for k in ['DIST_BACKEND', 'NCCL_P2P_DISABLE', 'NCCL_IB_DISABLE', 'NCCL_DEBUG', 'NCCL_CUMEM_ENABLE', 'NCCL_CUMEM_HOST_ENABLE', 'TORCH_NCCL_ASYNC_ERROR_HANDLING']})
run = subprocess.run(ddp_cmd, check=False, env=ddp_env)
if run.returncode != 0:
    raise RuntimeError(f'DDP training failed with return code {run.returncode}')

## This cell works only for the subset of data

In [ ]:
# Generate images and compare them to the saved ground-truth training subset.
# Columns: GT image, baseline MaskGen generation, critic-guided generation.
import json
import os
import textwrap
from PIL import Image
import matplotlib.pyplot as plt

from token_regrate.train_token_regret_ddp import generate_image_vq_batch, set_global_seed
from token_regrate.utils import load_trained_critic

if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
EXPERIMENT_CFG = globals().get('EXPERIMENT_CFG', CONFIG.experiment)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)
CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))
MODEL = globals().get('MODEL', CONFIG.model)
VIZ_CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))


def _viz_resolve_path(path_value):
    path_value = os.path.expanduser(str(path_value))
    if os.path.isabs(path_value):
        return path_value
    root = globals().get('PROJECT_ROOT', os.path.abspath('.'))
    return os.path.abspath(os.path.join(root, path_value))


def _load_viz_subset(manifest_path):
    manifest_path = _viz_resolve_path(manifest_path)
    if not os.path.isfile(manifest_path):
        raise FileNotFoundError(f'Used-training manifest not found: {manifest_path}')

    subset_root = os.path.dirname(manifest_path)
    rows, images, captions = [], [], []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            image_path = str(row.get('image', '')).strip()
            if image_path and not os.path.isabs(image_path):
                image_path = os.path.join(subset_root, image_path)
            if not os.path.isfile(image_path):
                raise FileNotFoundError(f'Subset image missing: {image_path}')
            with Image.open(image_path) as img:
                images.append(img.convert('RGB'))
            row['image_path'] = image_path
            rows.append(row)
            captions.append(str(row.get('caption', '')).strip())
    if not captions:
        raise RuntimeError(f'No examples found in {manifest_path}')
    return rows, images, captions


def _resolve_viz_checkpoint():
    if '_resolve_critic_checkpoint' in globals():
        try:
            return _resolve_critic_checkpoint(globals().get('CRITIC_CKPT_PATH', str(getattr(RUNTIME_CFG, 'resume_checkpoint', ''))))
        except FileNotFoundError:
            pass
    candidates = [
        globals().get('CRITIC_CKPT_PATH', ''),
        os.environ.get('TOKEN_REGRET_VIZ_CKPT', '').strip(),
        str(getattr(RUNTIME_CFG, 'resume_checkpoint', '')).strip(),
        os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_last.pt'),
    ]
    for candidate in candidates:
        if candidate:
            candidate = _viz_resolve_path(candidate)
            if os.path.isfile(candidate):
                return candidate
    return None


def _generate_viz_images(prompts, use_critic_head=False, critic=None, seed_offset=0):
    images = []
    batch_size = max(1, int(VIZ_BATCH_SIZE))
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        set_global_seed(int(VIZ_SEED) + int(seed_offset) + int(start))
        images.extend(
            generate_image_vq_batch(
                prompts=batch_prompts,
                model=maskgen_vq_generator,
                tokenizer=tatitok_vq_tokenizer,
                clip_tokenizer=clip_tokenizer,
                clip_encoder=clip_encoder,
                guidance_scale=float(TRAIN_CFG.train_guidance_scale),
                randomize_temperature=float(TRAIN_CFG.train_randomize_temperature),
                aesthetic_score=float(TRAIN_CFG.train_aesthetic_score),
                num_sample_steps=int(MODEL.sample_steps),
                use_regret_remask=bool(use_critic_head),
                critic=critic,
                remask_ratio=float(INF_CFG.remask_ratio),
                refine_start_step=int(INF_CFG.refine_start_step),
                refine_loops=int(INF_CFG.refine_loops),
                repair_greedy=bool(INF_CFG.repair_greedy),
            )
        )
    return images


VIZ_MANIFEST_PATH = _viz_resolve_path(
    os.path.join(str(EXPERIMENT_CFG.output_dir), 'used_training_images', 'manifest.jsonl')
)
VIZ_START_INDEX = max(0, int(os.environ.get('TOKEN_REGRET_VIZ_START_INDEX', '0')))
VIZ_NUM_IMAGES = max(1, int(os.environ.get('TOKEN_REGRET_VIZ_NUM_IMAGES', '8')))
VIZ_BATCH_SIZE = max(1, int(os.environ.get('TOKEN_REGRET_VIZ_BATCH_SIZE', '4')))
VIZ_SEED = int(os.environ.get('TOKEN_REGRET_VIZ_SEED', int(INF_CFG.seed)))
VIZ_SHOW_BASELINE = os.environ.get('TOKEN_REGRET_VIZ_SHOW_BASELINE', '1').lower() in {'1', 'true', 'yes', 'on'}
VIZ_SHOW_CRITIC = os.environ.get('TOKEN_REGRET_VIZ_SHOW_CRITIC', '1').lower() in {'1', 'true', 'yes', 'on'}

viz_rows_all, viz_gt_all, viz_captions_all = _load_viz_subset(VIZ_MANIFEST_PATH)
viz_end = min(len(viz_captions_all), VIZ_START_INDEX + VIZ_NUM_IMAGES)
viz_rows = viz_rows_all[VIZ_START_INDEX:viz_end]
viz_gt_images = viz_gt_all[VIZ_START_INDEX:viz_end]
viz_captions = viz_captions_all[VIZ_START_INDEX:viz_end]
if not viz_captions:
    raise RuntimeError(
        f'No images selected. start={VIZ_START_INDEX}, count={VIZ_NUM_IMAGES}, total={len(viz_captions_all)}'
    )

viz_device = next(maskgen_vq_generator.parameters()).device
maskgen_vq_generator = maskgen_vq_generator.to(viz_device).eval()
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(viz_device).eval()
clip_encoder = clip_encoder.to(viz_device).eval()

viz_critic = None
if VIZ_SHOW_CRITIC:
    viz_ckpt_path = _resolve_viz_checkpoint()
    if viz_ckpt_path is None:
        VIZ_SHOW_CRITIC = False
    elif 'critic_head' in globals() and int(getattr(critic_head, 'prompt_gap_topk', VIZ_CRITIC_PROMPT_GAP_TOPK)) == VIZ_CRITIC_PROMPT_GAP_TOPK:
        viz_critic = critic_head.to(viz_device).eval()
    else:
        viz_critic = load_trained_critic(
            viz_ckpt_path,
            maskgen_vq_generator,
            use_hidden=True,
            prompt_gap_topk=VIZ_CRITIC_PROMPT_GAP_TOPK,
        ).to(viz_device).eval()

baseline_images = _generate_viz_images(viz_captions, use_critic_head=False, critic=None) if VIZ_SHOW_BASELINE else []
critic_images = _generate_viz_images(viz_captions, use_critic_head=True, critic=viz_critic) if VIZ_SHOW_CRITIC else []

columns = [('GT', viz_gt_images)]
if VIZ_SHOW_BASELINE:
    columns.append(('Generated', baseline_images))
if VIZ_SHOW_CRITIC:
    columns.append(('Critic-guided', critic_images))

fig_width = 4.2 * len(columns)
fig_height = max(3.0, 3.6 * len(viz_captions))
fig, axes = plt.subplots(len(viz_captions), len(columns), figsize=(fig_width, fig_height), squeeze=False)

for row_idx, caption in enumerate(viz_captions):
    short_caption = textwrap.shorten(caption.replace('\n', ' '), width=92, placeholder='...')
    for col_idx, (title, images) in enumerate(columns):
        ax = axes[row_idx][col_idx]
        ax.imshow(images[row_idx])
        ax.axis('off')
        heading = title if row_idx == 0 else ''
        if col_idx == 0:
            ax.set_title(f'{heading}\n#{VIZ_START_INDEX + row_idx}: {short_caption}', fontsize=9, loc='left')
        else:
            ax.set_title(heading, fontsize=10)

plt.tight_layout()
VIZ_OUTPUT_PATH = os.path.join(os.path.dirname(VIZ_MANIFEST_PATH), 'generated_vs_gt.png')
fig.savefig(VIZ_OUTPUT_PATH, dpi=160, bbox_inches='tight')
plt.show()

VIZ_COMPARISON = {
    'rows': viz_rows,
    'captions': viz_captions,
    'gt_images': viz_gt_images,
    'baseline_images': baseline_images,
    'critic_images': critic_images,
    'output_path': VIZ_OUTPUT_PATH,
}


## Load Critic Head

In [ ]:
import glob
import json
import os
import re
from PIL import Image
if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
EXPERIMENT_CFG = globals().get('EXPERIMENT_CFG', CONFIG.experiment)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)
CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))

if '_resolve_project_path' not in globals():
    def _resolve_project_path(path_value):
        path_value = os.path.expanduser(str(path_value))
        return path_value if os.path.isabs(path_value) else os.path.abspath(path_value)

def _load_used_training_examples(manifest_path=None, load_images=True):
    manifest_path = manifest_path or os.path.join(str(EXPERIMENT_CFG.output_dir), 'used_training_images', 'manifest.jsonl')
    manifest_path = _resolve_project_path(manifest_path)
    if not os.path.isfile(manifest_path):
        return [], [], []

    subset_root = os.path.dirname(manifest_path)
    rows = []
    images = []
    captions = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            image_path = str(row.get('image', '')).strip()
            if image_path and not os.path.isabs(image_path):
                image_path = os.path.join(subset_root, image_path)
            row['image_path'] = image_path
            rows.append(row)
            captions.append(str(row.get('caption', '')).strip())
            if load_images and image_path and os.path.isfile(image_path):
                with Image.open(image_path) as img:
                    images.append(img.convert('RGB'))
    return rows, images, captions


def _resolve_critic_checkpoint(preferred=None, fallback='latest'):
    candidates = []
    env_path = os.environ.get('TOKEN_REGRET_CRITIC_CKPT', '').strip()
    if env_path:
        candidates.append(env_path)
    if preferred:
        candidates.append(preferred)
    candidates.extend([
        str(RUNTIME_CFG.resume_checkpoint),
        os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_last.pt'),
    ])
    for cand in candidates:
        if not cand:
            continue
        cand = _resolve_project_path(cand)
        if os.path.isfile(cand):
            return cand

    ckpts = glob.glob(os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_step_*.pt'))
    if os.path.isfile(os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_last.pt')):
        ckpts.append(os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_last.pt'))
    if ckpts:
        step_ckpts = [p for p in ckpts if re.search(r'critic_step_(\\d+)\\.pt$', os.path.basename(p))]
        if fallback == 'earliest' and step_ckpts:
            return min(step_ckpts, key=lambda p: int(re.search(r'critic_step_(\\d+)\\.pt$', os.path.basename(p)).group(1)))
        return max(ckpts, key=lambda p: (10**18 if os.path.basename(p) == 'critic_last.pt' else int(re.search(r'critic_step_(\\d+)\\.pt$', os.path.basename(p)).group(1))))
    raise FileNotFoundError(
        'No critic checkpoint found. Set TOKEN_REGRET_CRITIC_CKPT or save a checkpoint under '
        f'{_resolve_project_path(EXPERIMENT_CFG.output_dir)}.'
    )

CRITIC_CKPT_PATH = _resolve_critic_checkpoint()
critic_head = load_trained_critic(
    CRITIC_CKPT_PATH,
    maskgen_vq_generator,
    use_hidden=True,
    prompt_gap_topk=CRITIC_PROMPT_GAP_TOPK,
 )
model_device = next(maskgen_vq_generator.parameters()).device
critic_head = critic_head.to(model_device)
print(f'Loaded critic head from: {CRITIC_CKPT_PATH}')

# Checkpoint-vs-checkpoint comparison: token deltas + decoded image deltas


In [ ]:
# Checkpoint-vs-checkpoint comparison: token deltas + decoded image deltas
import glob
import math
import os
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from token_regrate.train_token_regret_ddp import (
    generate_wrapper,
    load_trained_critic,
    prepare_text_guidance,
    set_global_seed,
)

# Notebook cells may be run out of order, so rebuild config-backed defaults if needed.
if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
MODEL = globals().get('MODEL', CONFIG.model)
EXPERIMENT_CFG = globals().get('EXPERIMENT_CFG', CONFIG.experiment)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)

if '_resolve_project_path' not in globals():
    def _resolve_project_path(path_value):
        path_value = os.path.expanduser(str(path_value))
        return path_value if os.path.isabs(path_value) else os.path.abspath(path_value)

def _compare_list_checkpoints(output_dir=None):
    if '_list_critic_checkpoints' in globals():
        return _list_critic_checkpoints(output_dir)
    output_dir = _resolve_project_path(output_dir or EXPERIMENT_CFG.output_dir)
    paths = []
    last_path = os.path.join(output_dir, 'critic_last.pt')
    if os.path.isfile(last_path):
        paths.append(last_path)
    paths.extend(glob.glob(os.path.join(output_dir, 'critic_step_*.pt')))
    return sorted(set(paths))

def _compare_checkpoint_step(path):
    if '_checkpoint_step' in globals():
        return _checkpoint_step(path)
    match = re.search(r'critic_step_(\d+)\.pt$', os.path.basename(str(path)))
    if match:
        return int(match.group(1))
    return 10**18 if os.path.basename(str(path)) == 'critic_last.pt' else -1

def _resolve_compare_checkpoint(preferred=None, fallback='latest'):
    if preferred:
        preferred = _resolve_project_path(preferred)
        if os.path.isfile(preferred):
            return preferred

    ckpts = _compare_list_checkpoints(EXPERIMENT_CFG.output_dir)
    step_ckpts = [p for p in ckpts if re.search(r'critic_step_(\d+)\.pt$', os.path.basename(p))]
    if fallback == 'earliest' and step_ckpts:
        return min(step_ckpts, key=_compare_checkpoint_step)
    if ckpts:
        return max(ckpts, key=_compare_checkpoint_step)
    raise FileNotFoundError(
        'No critic checkpoint found. Set TOKEN_REGRET_COMPARE_CKPT_A / TOKEN_REGRET_COMPARE_CKPT_B, '
        f'or save checkpoints under {_resolve_project_path(EXPERIMENT_CFG.output_dir)}.'
    )

# Pick two checkpoints to compare. Env vars override the config-backed defaults.
CKPT_A_PREFERRED = os.environ.get('TOKEN_REGRET_COMPARE_CKPT_A', '').strip() or os.path.join(
    str(EXPERIMENT_CFG.output_dir),
    'critic_step_1.pt',
)
CKPT_B_PREFERRED = os.environ.get('TOKEN_REGRET_COMPARE_CKPT_B', '').strip() or str(RUNTIME_CFG.resume_checkpoint)
CKPT_A = _resolve_compare_checkpoint(CKPT_A_PREFERRED, fallback='earliest')
CKPT_B = _resolve_compare_checkpoint(CKPT_B_PREFERRED, fallback='latest')

# Reuse prompt list from previous cells if available; otherwise load fresh prompts.
COMPARE_PROMPT_FILE = os.path.abspath('geneval/prompts/generation_prompts.txt')
COMPARE_NUM_PROMPTS = 3
COMPARE_START_INDEX = 90

# Keep settings aligned with the config-backed visual test cell.
COMPARE_SEED = int(INF_CFG.seed)
COMPARE_GUIDANCE_SCALE = float(TRAIN_CFG.train_guidance_scale)
COMPARE_RANDOMIZE_TEMPERATURE = float(TRAIN_CFG.train_randomize_temperature)
COMPARE_AESTHETIC_SCORE = float(TRAIN_CFG.train_aesthetic_score)
COMPARE_NUM_SAMPLE_STEPS = int(MODEL.sample_steps)
COMPARE_REMASK_RATIO = float(INF_CFG.remask_ratio)
COMPARE_REFINE_LOOPS = int(INF_CFG.refine_loops)
COMPARE_REFINE_START_STEP = int(INF_CFG.refine_start_step)
COMPARE_REPAIR_GREEDY = bool(INF_CFG.repair_greedy)
COMPARE_CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))

if 'trace_prompts' in globals() and isinstance(trace_prompts, list) and len(trace_prompts) > 0:
    prompts_eval = trace_prompts[:max(1, int(COMPARE_NUM_PROMPTS))]
elif 'prompts' in globals() and isinstance(prompts, list) and len(prompts) > 0:
    prompts_eval = prompts[:max(1, int(COMPARE_NUM_PROMPTS))]
else:
    if not os.path.isfile(COMPARE_PROMPT_FILE):
        raise FileNotFoundError(f'GenEval prompt file not found: {COMPARE_PROMPT_FILE}')
    with open(COMPARE_PROMPT_FILE, 'r') as f:
        all_prompts = [line.strip() for line in f if line.strip()]
    s = max(0, int(COMPARE_START_INDEX))
    e = min(len(all_prompts), s + max(1, int(COMPARE_NUM_PROMPTS)))
    prompts_eval = all_prompts[s:e]

if len(prompts_eval) == 0:
    raise RuntimeError('No prompts selected for checkpoint comparison.')

model_device = next(maskgen_vq_generator.parameters()).device

critic_a = load_trained_critic(
    CKPT_A,
    maskgen_vq_generator,
    use_hidden=True,
    prompt_gap_topk=COMPARE_CRITIC_PROMPT_GAP_TOPK,
).to(model_device)
critic_b = load_trained_critic(
    CKPT_B,
    maskgen_vq_generator,
    use_hidden=True,
    prompt_gap_topk=COMPARE_CRITIC_PROMPT_GAP_TOPK,
).to(model_device)
critic_a.eval()
critic_b.eval()


def _generate_tokens_with_critic(critic):
    set_global_seed(COMPARE_SEED)
    toks = generate_wrapper(
        model=maskgen_vq_generator,
        captions=prompts_eval,
        clip_tokenizer=clip_tokenizer,
        clip_encoder=clip_encoder,
        guidance_scale=COMPARE_GUIDANCE_SCALE,
        randomize_temperature=COMPARE_RANDOMIZE_TEMPERATURE,
        sample_aesthetic_score=COMPARE_AESTHETIC_SCORE,
        num_sample_steps=COMPARE_NUM_SAMPLE_STEPS,
        use_critic_head=True,
        critic=critic,
        remask_ratio=COMPARE_REMASK_RATIO,
        refine_loops=COMPARE_REFINE_LOOPS,
        refine_start_step=COMPARE_REFINE_START_STEP,
        repair_greedy=COMPARE_REPAIR_GREEDY,
        critic_use_hidden=True,
    )
    return toks.to(model_device)


def _decode_tokens(tokens):
    text_guidance = prepare_text_guidance(prompts_eval, clip_tokenizer, clip_encoder, model_device)
    img = tatitok_vq_tokenizer.decode_tokens(tokens, text_guidance)
    img = torch.clamp(img, 0.0, 1.0)
    arr = (img * 255.0).permute(0, 2, 3, 1).to('cpu', dtype=torch.uint8).numpy()
    return arr


print('Comparing checkpoints:')
print(f'  A: {CKPT_A}')
print(f'  B: {CKPT_B}')
print(f'Prompts: {len(prompts_eval)}')
print('Generation settings:')
print(f'  guidance_scale={COMPARE_GUIDANCE_SCALE}')
print(f'  randomize_temperature={COMPARE_RANDOMIZE_TEMPERATURE}')
print(f'  aesthetic_score={COMPARE_AESTHETIC_SCORE}')
print(f'  num_sample_steps={COMPARE_NUM_SAMPLE_STEPS}')
print(f'  remask_ratio={COMPARE_REMASK_RATIO}')
print('  remask_score_threshold=disabled (pure top-k)')
print(f'  refine_loops={COMPARE_REFINE_LOOPS}')
print(f'  refine_start_step={COMPARE_REFINE_START_STEP}')
print(f'  repair_greedy={COMPARE_REPAIR_GREEDY}')
print(f'  critic_prompt_gap_topk={COMPARE_CRITIC_PROMPT_GAP_TOPK}')

# Run both with the same seed for fair token-level comparison.
tokens_a = _generate_tokens_with_critic(critic_a)
tokens_b = _generate_tokens_with_critic(critic_b)

# Token-level stats
token_diff = tokens_a.ne(tokens_b)
changed_tokens = token_diff.sum(dim=-1).to('cpu').numpy()
seq_len = int(tokens_a.shape[1])
changed_ratio = (changed_tokens / float(seq_len)) * 100.0

print()
print('Token differences per sample:')
for i in range(tokens_a.shape[0]):
    print(f'  sample {i}: changed {int(changed_tokens[i])}/{seq_len} ({changed_ratio[i]:.2f}%)')
print(f'Average changed tokens: {float(changed_tokens.mean()):.2f}/{seq_len} ({float(changed_ratio.mean()):.2f}%)')

# Decode and compare images.
imgs_a = _decode_tokens(tokens_a)
imgs_b = _decode_tokens(tokens_b)

abs_diff = np.abs(imgs_a.astype(np.int16) - imgs_b.astype(np.int16)).astype(np.uint8)
img_mae = abs_diff.mean(axis=(1, 2, 3))
img_max = abs_diff.max(axis=(1, 2, 3))

print()
print('Image differences per sample:')
for i in range(len(prompts_eval)):
    print(f'  sample {i}: MAE={img_mae[i]:.2f} px, MAX={int(img_max[i])} px')

# Token-diff map (if token sequence is square) for easy visualization.
side = int(round(math.sqrt(seq_len)))
has_square_grid = (side * side == seq_len)
if has_square_grid:
    token_diff_map = token_diff.to('cpu').numpy().reshape(len(prompts_eval), side, side)
else:
    token_diff_map = None

# Visualize A, B, abs image diff, and token-diff map.
n = len(prompts_eval)
cols = 4 if has_square_grid else 3
fig, axes = plt.subplots(n, cols, figsize=(4.2 * cols, 3.6 * n))
axes = np.array(axes).reshape(n, cols)

for i in range(n):
    p = prompts_eval[i].replace(chr(10), ' ').strip()
    if len(p) > 80:
        p = p[:77] + '...'

    axes[i, 0].imshow(imgs_a[i])
    axes[i, 0].axis('off')
    axes[i, 0].set_title('A: ' + os.path.basename(CKPT_A) + chr(10) + p, fontsize=9)

    axes[i, 1].imshow(imgs_b[i])
    axes[i, 1].axis('off')
    axes[i, 1].set_title('B: ' + os.path.basename(CKPT_B), fontsize=9)

    # Mean-over-channel diff heatmap.
    axes[i, 2].imshow(abs_diff[i].mean(axis=2), cmap='inferno')
    axes[i, 2].axis('off')
    axes[i, 2].set_title(
        '|A-B| heatmap' + chr(10) + f'MAE={img_mae[i]:.2f}, token Δ={changed_ratio[i]:.2f}%',
        fontsize=9,
    )

    if has_square_grid:
        axes[i, 3].imshow(token_diff_map[i], cmap='magma', vmin=0, vmax=1, interpolation='nearest')
        axes[i, 3].axis('off')
        axes[i, 3].set_title(f'Token changed map ({side}x{side})', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Critic selection lift diagnostic: are remasked tokens better than random?
# This uses a tiny real training batch and computes true counterfactual regret for:
#   1. tokens selected by critic top-k predicted regret
#   2. tokens selected uniformly at random from the same visible-token candidate set
#   3. optional critic bottom-k tokens as a sanity check
# Meaningful behavior should show: top_mean > random_mean, positive correlation, and top_positive_frac > random_positive_frac.

import math
import os
import torch
from token_regrate.dataset import TrainingDatasetPipeline
from token_regrate.train_token_regret_ddp import (
    _resolve_schedule_remask_count,
    build_rollout_state,
    compute_counterfactual_regret,
    forward_maskgen,
    masked_pearson_corr,
    open_clip_text_encoding,
    select_topk_token_positions,
    set_global_seed,
)
from token_regrate.utils import images_to_tokens, load_trained_critic

if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
DATA_CFG = globals().get('DATA_CFG', CONFIG.dataset)
EXPERIMENT_CFG = globals().get('EXPERIMENT_CFG', CONFIG.experiment)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)
MODEL = globals().get('MODEL', CONFIG.model)
DIAG_CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))

if 'critic_head' not in globals() or int(getattr(critic_head, 'prompt_gap_topk', DIAG_CRITIC_PROMPT_GAP_TOPK)) != DIAG_CRITIC_PROMPT_GAP_TOPK:
    diag_ckpt_path = globals().get('CRITIC_CKPT_PATH', str(RUNTIME_CFG.resume_checkpoint))
    if '_resolve_critic_checkpoint' in globals():
        diag_ckpt_path = _resolve_critic_checkpoint(diag_ckpt_path)
    critic_head = load_trained_critic(
        diag_ckpt_path,
        maskgen_vq_generator,
        use_hidden=True,
        prompt_gap_topk=DIAG_CRITIC_PROMPT_GAP_TOPK,
    )

model_device = next(maskgen_vq_generator.parameters()).device
critic_head = critic_head.to(model_device).eval()
maskgen_vq_generator = maskgen_vq_generator.to(model_device).eval()
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(model_device).eval()
clip_encoder = clip_encoder.to(model_device).eval()

# Keep this deliberately small; increase after the quick check works.
DIAG_BATCH_SIZE = 32
DIAG_MAX_K = 16
DIAG_RANDOM_TRIALS = 3
DIAG_CF_CHUNK_SIZE = 16
DIAG_ROLLOUT_STEPS = int(getattr(TRAIN_CFG, 'counterfactual_rollout_steps', 1))
DIAG_TRAIN_REMASK_RATIO = float(getattr(TRAIN_CFG, 'train_remask_ratio', getattr(INF_CFG, 'remask_ratio', 0.05)))
DIAG_REFINE_LOOPS = int(getattr(TRAIN_CFG, 'refine_loops', getattr(INF_CFG, 'refine_loops', 1)))
DIAG_REFINE_START_STEP = int(getattr(TRAIN_CFG, 'train_refine_start_step', getattr(INF_CFG, 'refine_start_step', 10)))
DIAG_TRAIN_REPAIR_GREEDY = bool(getattr(TRAIN_CFG, 'train_repair_greedy', getattr(INF_CFG, 'repair_greedy', False)))
DIAG_COUNTERFACTUAL_REPAIR_GREEDY = bool(getattr(TRAIN_CFG, 'counterfactual_repair_greedy', getattr(INF_CFG, 'repair_greedy', False)))
DIAG_COUNTERFACTUAL_UTILITY = str(getattr(TRAIN_CFG, 'counterfactual_utility', 'token_ce'))
DIAG_COUNTERFACTUAL_WINDOW_RADIUS = int(getattr(TRAIN_CFG, 'counterfactual_window_radius', 0))
DIAG_COUNTERFACTUAL_CONTRAST_NEGATIVES = int(getattr(TRAIN_CFG, 'counterfactual_contrast_negatives', 2))
DIAG_COUNTERFACTUAL_CONTRAST_TEMPERATURE = float(getattr(TRAIN_CFG, 'counterfactual_contrast_temperature', 1.0))
DIAG_COUNTERFACTUAL_CONTRAST_MODE = str(getattr(TRAIN_CFG, 'counterfactual_contrast_mode', 'nce'))
DIAG_FIXED_ROLLOUT_STEP = int(getattr(TRAIN_CFG, 'fixed_rollout_step', -1))
DIAG_USE_CRITIC_REPLAY_STATE = bool(getattr(TRAIN_CFG, 'use_target_critic_replay', False))
DIAG_SEED = int(getattr(TRAIN_CFG, 'seed', getattr(INF_CFG, 'seed', 42)))
DIAG_USE_SAVED_EXAMPLES = False

set_global_seed(DIAG_SEED)

diag_images = globals().get('USED_TRAINING_IMAGES', []) if DIAG_USE_SAVED_EXAMPLES else []
diag_captions = globals().get('USED_TRAINING_CAPTIONS', []) if DIAG_USE_SAVED_EXAMPLES else []

if (not diag_images or not diag_captions) and '_load_used_training_examples' in globals():
    _, diag_images, diag_captions = _load_used_training_examples(load_images=True)

if diag_images and diag_captions:
    diag_images = diag_images[:max(1, int(DIAG_BATCH_SIZE))]
    diag_captions = diag_captions[:len(diag_images)]
    print(f'Using {len(diag_captions)} saved fixed training examples for diagnostic.')
else:
    diag_dataset = TrainingDatasetPipeline(
        mode=str(DATA_CFG.mode),
        sources=str(DATA_CFG.source),
        cc12m_cache_dir=str(DATA_CFG.cc12m_cache_dir),
        cc12m_cache_images=not bool(DATA_CFG.disable_cc12m_cache),
        cc12m_loader_workers=min(4, int(DATA_CFG.cc12m_loader_workers)),
        cc12m_max_pending=min(32, int(DATA_CFG.cc12m_loader_max_pending)),
    )
    diag_images, diag_captions = next(diag_dataset.iter_batches(batch_size=DIAG_BATCH_SIZE, rank=0, world_size=1))

diag_batch_size = len(diag_captions)
if diag_batch_size == 0:
    raise RuntimeError('Diagnostic dataset produced an empty batch.')
if str(DIAG_COUNTERFACTUAL_UTILITY).endswith('_contrast') and diag_batch_size < 2:
    raise RuntimeError('Contrastive counterfactual diagnostics require at least two prompts in the batch.')

print('Diagnostic prompts:')
for i, caption in enumerate(diag_captions):
    short = caption.replace('\n', ' ').strip()
    print(f'  {i}: {short[:140]}')

gt_tokens = images_to_tokens(tatitok_vq_tokenizer, diag_images).to(model_device, non_blocking=True)
condition, condition_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, diag_captions)
condition = condition.to(model_device, non_blocking=True)
condition_pooled = condition_pooled.to(model_device, non_blocking=True)
none_condition, none_condition_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, [''])
none_condition = none_condition.repeat(diag_batch_size, 1, 1).to(model_device, non_blocking=True)
none_condition_pooled = none_condition_pooled.repeat(diag_batch_size, 1, 1).to(model_device, non_blocking=True)

with torch.no_grad():
    z_t, timesteps, step_indices = build_rollout_state(
        model=maskgen_vq_generator,
        condition=condition,
        condition_pooled=condition_pooled,
        none_condition=none_condition,
        none_condition_pooled=none_condition_pooled,
        device=model_device,
        batch_size=diag_batch_size,
        guidance_scale=float(TRAIN_CFG.train_guidance_scale),
        randomize_temperature=float(TRAIN_CFG.train_randomize_temperature),
        sample_aesthetic_score=float(TRAIN_CFG.train_aesthetic_score),
        num_sample_steps=int(MODEL.sample_steps),
        softmax_temperature_annealing=True,
        refine_softmax_temperature=0.7,
        guidance_decay='cosine',
        guidance_decay_scale_pow=1.0,
        prob_sorting=True,
        refine_loops=DIAG_REFINE_LOOPS,
        refine_start_step=DIAG_REFINE_START_STEP,
        repair_greedy=DIAG_TRAIN_REPAIR_GREEDY,
        fixed_step=DIAG_FIXED_ROLLOUT_STEP if DIAG_FIXED_ROLLOUT_STEP >= 0 else None,
    )

    logits_orig, hidden_orig, text_feat, cond_logits_orig, uncond_logits_orig = forward_maskgen(
        model=maskgen_vq_generator,
        input_ids=z_t,
        condition=condition,
        condition_pooled=condition_pooled,
        ratio=timesteps,
        guidance_scale=float(TRAIN_CFG.train_guidance_scale),
        sample_aesthetic_score=float(TRAIN_CFG.train_aesthetic_score),
        guidance_decay='cosine',
        guidance_decay_scale_pow=1.0,
        none_condition=none_condition,
        none_condition_pooled=none_condition_pooled,
        return_cfg_parts=True,
    )
    prompt_logits_delta_orig = cond_logits_orig - uncond_logits_orig if uncond_logits_orig is not None else None
    pred_regret = critic_head(
        hidden_states=hidden_orig,
        logits=logits_orig,
        timesteps=timesteps,
        text_features=text_feat,
        prompt_logits_delta=prompt_logits_delta_orig,
    )

candidate_mask = z_t.ne(int(maskgen_vq_generator.mask_token_id))
visible_counts = candidate_mask.sum(dim=-1)
schedule_k = _resolve_schedule_remask_count(
    maskgen_vq_generator,
    timesteps,
    budget_fraction=DIAG_TRAIN_REMASK_RATIO,
)
eval_k = min(int(schedule_k), int(DIAG_MAX_K), int(visible_counts.min().item()))
if eval_k <= 0:
    raise RuntimeError(
        f'No visible candidate tokens to evaluate. visible_counts={visible_counts.tolist()}, schedule_k={schedule_k}'
    )

critic_idx, critic_valid = select_topk_token_positions(
    pred_regret,
    eval_k,
    candidate_mask=candidate_mask,
    min_tokens=1,
)

# Bottom-k should usually have lower true regret than top-k if the critic has learned a useful ordering.
bottom_scores = -pred_regret
bottom_idx, bottom_valid = select_topk_token_positions(
    bottom_scores,
    eval_k,
    candidate_mask=candidate_mask,
    min_tokens=1,
)

random_idx_parts = []
random_valid_parts = []
for trial in range(int(DIAG_RANDOM_TRIALS)):
    rand_scores = torch.rand_like(pred_regret).masked_fill(~candidate_mask, float('-inf'))
    rand_idx = rand_scores.topk(k=eval_k, dim=-1).indices
    rand_valid = candidate_mask.gather(dim=-1, index=rand_idx)
    random_idx_parts.append(rand_idx)
    random_valid_parts.append(rand_valid)
random_idx = torch.cat(random_idx_parts, dim=-1)
random_valid = torch.cat(random_valid_parts, dim=-1)

selected_idx = torch.cat([critic_idx, random_idx, bottom_idx], dim=-1)
selected_valid = torch.cat([critic_valid, random_valid, bottom_valid], dim=-1)

true_regrets = compute_counterfactual_regret(
    model=maskgen_vq_generator,
    gt_tokens=gt_tokens,
    z_t=z_t,
    condition=condition,
    condition_pooled=condition_pooled,
    timesteps=timesteps,
    step_indices=step_indices,
    selected_idx=selected_idx,
    selected_valid=selected_valid,
    sample_aesthetic_score=float(TRAIN_CFG.train_aesthetic_score),
    counterfactual_chunk_size=int(DIAG_CF_CHUNK_SIZE),
    counterfactual_rollout_steps=int(DIAG_ROLLOUT_STEPS),
    counterfactual_utility=DIAG_COUNTERFACTUAL_UTILITY,
    counterfactual_contrast_negatives=DIAG_COUNTERFACTUAL_CONTRAST_NEGATIVES,
    counterfactual_contrast_temperature=DIAG_COUNTERFACTUAL_CONTRAST_TEMPERATURE,
    counterfactual_contrast_mode=DIAG_COUNTERFACTUAL_CONTRAST_MODE,
    num_sample_steps=int(MODEL.sample_steps),
    guidance_scale=float(TRAIN_CFG.train_guidance_scale),
    randomize_temperature=float(TRAIN_CFG.train_randomize_temperature),
    refine_softmax_temperature=0.7,
    guidance_decay='cosine',
    guidance_decay_scale_pow=1.0,
    softmax_temperature_annealing=False,
    prob_sorting=True,
    repair_greedy=DIAG_COUNTERFACTUAL_REPAIR_GREEDY,
    none_condition=none_condition,
    none_condition_pooled=none_condition_pooled,
)

critic_slice = slice(0, eval_k)
random_slice = slice(eval_k, eval_k + eval_k * int(DIAG_RANDOM_TRIALS))
bottom_slice = slice(eval_k + eval_k * int(DIAG_RANDOM_TRIALS), eval_k + eval_k * int(DIAG_RANDOM_TRIALS) + eval_k)

critic_true = true_regrets[:, critic_slice]
critic_pred = pred_regret.gather(dim=-1, index=critic_idx)
critic_keep = selected_valid[:, critic_slice]
random_true = true_regrets[:, random_slice]
random_keep = selected_valid[:, random_slice]
bottom_true = true_regrets[:, bottom_slice]
bottom_pred = pred_regret.gather(dim=-1, index=bottom_idx)
bottom_keep = selected_valid[:, bottom_slice]

all_eval_pred = pred_regret.gather(dim=-1, index=selected_idx)
all_corr = masked_pearson_corr(all_eval_pred, true_regrets, selected_valid)

def _masked_mean(x, keep):
    return float(x[keep].mean().item()) if keep.any() else float('nan')

def _masked_std(x, keep):
    return float(x[keep].std(unbiased=False).item()) if keep.any() else float('nan')

def _positive_frac(x, keep):
    return float(x[keep].gt(0).float().mean().item()) if keep.any() else float('nan')

critic_mean = _masked_mean(critic_true, critic_keep)
random_mean = _masked_mean(random_true, random_keep)
bottom_mean = _masked_mean(bottom_true, bottom_keep)
critic_pos = _positive_frac(critic_true, critic_keep)
random_pos = _positive_frac(random_true, random_keep)
bottom_pos = _positive_frac(bottom_true, bottom_keep)
selection_lift = critic_mean - random_mean
bottom_gap = critic_mean - bottom_mean

print('\nCritic selection lift diagnostic')
print(f'  batch_size={diag_batch_size}')
print(f'  step_indices={step_indices.detach().cpu().tolist()}')
print(f'  timesteps={[round(float(x), 4) for x in timesteps.detach().cpu().tolist()]}')
print(f'  visible_counts={visible_counts.detach().cpu().tolist()}')
print(f'  schedule_k={int(schedule_k)} eval_k={int(eval_k)} random_trials={int(DIAG_RANDOM_TRIALS)}')
print(f'  rollout_steps_for_true_regret={int(DIAG_ROLLOUT_STEPS)}')
print(f'  counterfactual_utility={DIAG_COUNTERFACTUAL_UTILITY} window_radius={DIAG_COUNTERFACTUAL_WINDOW_RADIUS}')
print(f'  contrast_negatives={DIAG_COUNTERFACTUAL_CONTRAST_NEGATIVES} contrast_temperature={DIAG_COUNTERFACTUAL_CONTRAST_TEMPERATURE} contrast_mode={DIAG_COUNTERFACTUAL_CONTRAST_MODE}')
print(f'  remask_ratio={DIAG_TRAIN_REMASK_RATIO} refine_loops={DIAG_REFINE_LOOPS} refine_start_step={DIAG_REFINE_START_STEP}')
print(f'  remask_score_threshold=disabled critic_prompt_gap_topk={DIAG_CRITIC_PROMPT_GAP_TOPK}')
print(f'  train_repair_greedy={DIAG_TRAIN_REPAIR_GREEDY} counterfactual_repair_greedy={DIAG_COUNTERFACTUAL_REPAIR_GREEDY}')
print(f'  use_target_replay_state={DIAG_USE_CRITIC_REPLAY_STATE} fixed_rollout_step={DIAG_FIXED_ROLLOUT_STEP}')
print(f'  critic_true_mean={critic_mean:.6f} std={_masked_std(critic_true, critic_keep):.6f} positive_frac={critic_pos:.4f}')
print(f'  random_true_mean={random_mean:.6f} std={_masked_std(random_true, random_keep):.6f} positive_frac={random_pos:.4f}')
print(f'  bottom_true_mean={bottom_mean:.6f} std={_masked_std(bottom_true, bottom_keep):.6f} positive_frac={bottom_pos:.4f}')
print(f'  selection_lift_vs_random={selection_lift:.6f}')
print(f'  top_minus_bottom_true_regret={bottom_gap:.6f}')
print(f'  pred_true_corr_on_evaluated_positions={float(all_corr.item()):.6f}')
print(f'  critic_pred_mean={_masked_mean(critic_pred, critic_keep):.6f}')
print(f'  bottom_pred_mean={_masked_mean(bottom_pred, bottom_keep):.6f}')

if selection_lift > 0 and bottom_gap > 0 and float(all_corr.item()) > 0:
    print('Verdict: PASS-ish. The critic-selected tokens have higher measured regret than random/bottom on this tiny batch.')
else:
    print('Verdict: FAIL / not convincing. The critic selections do not beat random/bottom on this tiny batch.')
    print('This means the visual token changes are probably not meaningfully targeted yet, or the checkpoint is too early/noisy.')